In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# Cell 1: GPU check
!nvidia-smi


Mon Mar 23 14:10:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Cell 2: system deps
!apt-get -qq update
!apt-get -qq install -y espeak-ng ffmpeg sox libsndfile1


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../00-libpcaudio0_1.1-6build2_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.1-6build2) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../01-libsonic0_0.2.0-11build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-11build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../02-espeak-ng-data_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../03-libespeak-ng1_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.50+dfsg-10ubuntu0.1) ...
Sel

In [5]:
# Cell 3: python deps + Matcha
!pip -q install --upgrade pip
!pip -q install pandas pyarrow soundfile librosa datasets huggingface_hub
!git clone https://github.com/shivammehta25/Matcha-TTS.git
%cd Matcha-TTS
!pip -q install -e .


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.1 MB/s eta 0:00:0000:01
Cloning into 'Matcha-TTS'...
remote: Enumerating objects: 1082, done.
remote: Counting objects: 100% (553/553), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 1082 (delta 447), reused 355 (delta 355), pack-reused 529 (from 3)
Receiving objects: 100% (1082/1082), 64.10 MiB | 37.11 MiB/s, done.
Resolving deltas: 100% (536/536), done.
/kaggle/working/Matcha-TTS
  error: subprocess-exited-with-error
  
  × installing build dependencies did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
ERROR: Failed to build 'file:///kaggle/working/Matcha-TTS' when installing build dependencies


In [6]:
# Cell 1 (bash)
%cd /kaggle/working/Matcha-TTS


/kaggle/working/Matcha-TTS


In [7]:
# Cell 2 (python)
import sys
from pathlib import Path

print("Python:", sys.version)
p = Path("pyproject.toml")
t = p.read_text()

if sys.version_info >= (3, 12):
    t = t.replace("numpy==1.24.3", "numpy>=1.26.4")
    t = t.replace("cython==0.29.35", "cython>=0.29.35")
    p.write_text(t)
    print("Patched pyproject.toml for Python >= 3.12")
else:
    print("No pyproject patch needed")


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Patched pyproject.toml for Python >= 3.12


In [8]:
# Cell 3 (bash)
!pip install -U pip setuptools wheel packaging
!pip install -U "numpy>=1.24.3" "cython>=0.29.35"
!pip install --no-build-isolation -e . -v


  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.7 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 5.7 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 7.4 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2━━━━━━ 0/2 [numpy]
  Attempting uninstall: cython━━━━━━━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: Cython 3.0.120m 0/2 [numpy]
    Uninstalling Cython-3.0.12:━━━━━━━━━━━━━ 0/2 [numpy]
      Successfully uninstalled Cython-3.0.1290m━━━━━━━━━━━━━━━━━━━ 1/2 [cython]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cython]2m1/2 [cython]
ERROR: pip's depen

In [9]:
# Cell 4: verify CUDA + create work dirs
import os, torch
print("CUDA:", torch.cuda.is_available())
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

os.makedirs("/kaggle/working/hi_ta_matcha/data", exist_ok=True)
os.makedirs("/kaggle/working/hi_ta_matcha/exp", exist_ok=True)
print("Work dir ready.")


CUDA: True
GPU : Tesla T4
Work dir ready.


In [10]:
!pip -q install -U huggingface_hub datasets pyarrow


In [11]:
import os
from huggingface_hub import login, HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")

assert token, "HF_TOKEN not found in Kaggle Secrets"

login(token=token)
api = HfApi()

repo_id = "ai4bharat/indicvoices_r"
files = api.list_repo_files(repo_id=repo_id, repo_type="dataset", token=token)

tamil_files = [f for f in files if f.startswith("Tamil/") and f.endswith(".parquet")]
print("Tamil parquet files:", len(tamil_files))
print(tamil_files[:10])


Tamil parquet files: 138
['Tamil/test-00000-of-00001.parquet', 'Tamil/train-00000-of-00137.parquet', 'Tamil/train-00001-of-00137.parquet', 'Tamil/train-00002-of-00137.parquet', 'Tamil/train-00003-of-00137.parquet', 'Tamil/train-00004-of-00137.parquet', 'Tamil/train-00005-of-00137.parquet', 'Tamil/train-00006-of-00137.parquet', 'Tamil/train-00007-of-00137.parquet', 'Tamil/train-00008-of-00137.parquet']


In [12]:
# Download a pilot subset first (e.g., 8 shards)
out_dir = "/kaggle/working/ivr_tamil"
download_list = tamil_files[:8]

for f in download_list:
    p = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=f,
        token=token,
        local_dir=out_dir,
        local_dir_use_symlinks=False,
    )
    print("Downloaded:", p)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Tamil/test-00000-of-00001.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

Downloaded: /kaggle/working/ivr_tamil/Tamil/test-00000-of-00001.parquet


Tamil/train-00000-of-00137.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

Downloaded: /kaggle/working/ivr_tamil/Tamil/train-00000-of-00137.parquet


...

Downloaded: /kaggle/working/ivr_tamil/Tamil/train-00004-of-00137.parquet


Tamil/train-00005-of-00137.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

Downloaded: /kaggle/working/ivr_tamil/Tamil/train-00005-of-00137.parquet


Tamil/train-00006-of-00137.parquet:   0%|          | 0.00/445M [00:00<?, ?B/s]

Downloaded: /kaggle/working/ivr_tamil/Tamil/train-00006-of-00137.parquet


In [13]:
!df -h /kaggle/working



Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  3.5G   17G  18% /kaggle/working


In [14]:
import os, glob, pandas as pd, random

random.seed(42)
p = sorted(glob.glob("/kaggle/working/ivr_tamil/Tamil/train-*.parquet"))[0]
df = pd.read_parquet(p)

text_col = "normalized" if "normalized" in df.columns else ("text" if "text" in df.columns else "verbatim")
assert text_col in df.columns, "No text column"

assert "audio" in df.columns, "No audio column in this shard"

out_audio = "/kaggle/working/hi_ta_matcha/audio_ta_test"
out_list = "/kaggle/working/Matcha-TTS/data/filelists"
os.makedirs(out_audio, exist_ok=True)
os.makedirs(out_list, exist_ok=True)

rows = []
n = 0
for _, r in df.iterrows():
    if n >= 200:
        break
    txt = str(r[text_col]).strip()
    if not txt:
        continue

    a = r["audio"]
    if not isinstance(a, dict):
        continue
    b = a.get("bytes", None)
    if not isinstance(b, (bytes, bytearray)):
        continue

    ext = ".wav"
    ap = a.get("path", "")
    if isinstance(ap, str) and "." in ap:
        e = os.path.splitext(ap)[1].lower()
        if e in [".wav", ".flac", ".ogg", ".mp3", ".m4a"]:
            ext = e

    wavp = os.path.join(out_audio, f"ta_test_{n:05d}{ext}")
    with open(wavp, "wb") as f:
        f.write(b)

    txt = " ".join(txt.replace("|", " ").split())
    rows.append((wavp, txt))
    n += 1

print("extracted:", len(rows))

train_txt = os.path.join(out_list, "ta_train_test.txt")
with open(train_txt, "w", encoding="utf-8") as f:
    for ap, tx in rows:
        f.write(f"{ap}|{tx}\n")

print("filelist:", train_txt)
with open(train_txt, "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().rstrip())


extracted: 200
filelist: /kaggle/working/Matcha-TTS/data/filelists/ta_train_test.txt
/kaggle/working/hi_ta_matcha/audio_ta_test/ta_test_00000.wav|நான் எதனா வந்துட்டு தப்பாக சொல்லியிருந்தா என்னை மன்னிச்சுடுங்க ஆனால் வந்துட்டு இனிமேல் யாரை கூப்பிட்டாலுமே கரெக்ட் டைமுக்கு போய் அவங்கள கொண்டு போய் விட்டுருங்க ஏன்னால் அவங்களுக்கு அதுவே வாழ்க்கையாக கூட அமையலாம் சரிங்களா அண்ணா ஓகே அண்ணா
/kaggle/working/hi_ta_matcha/audio_ta_test/ta_test_00001.wav|வளர்ந்து வரும் உலகம்
/kaggle/working/hi_ta_matcha/audio_ta_test/ta_test_00002.wav|நாளுக்கு நாள் உலகம் மாறிக்கிட்டு தான் இருக்குது என்னத்தை சொல்கிறது
/kaggle/working/hi_ta_matcha/audio_ta_test/ta_test_00003.wav|மாறிக்கிட்டு தான் இருக்குது உலகம்
/kaggle/working/hi_ta_matcha/audio_ta_test/ta_test_00004.wav|செய்தி தாள்களேயும் நியூஸ்களேயும்


In [15]:
import os, glob, random, re
import pandas as pd

random.seed(42)

def to_float_safe(x, default=None):
    if x is None:
        return default
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return default
    m = re.search(r"-?\d+(?:\.\d+)?", s)  # handles "tensor(0.)"
    return float(m.group(0)) if m else default

parquets = sorted(glob.glob("/kaggle/working/ivr_tamil/Tamil/train-*.parquet"))
print("shards:", len(parquets))

audio_dir = "/kaggle/working/hi_ta_matcha/audio_ta_pilot"
list_dir = "/kaggle/working/Matcha-TTS/data/filelists"
os.makedirs(audio_dir, exist_ok=True)
os.makedirs(list_dir, exist_ok=True)

TARGET_UTTS = 12000
rows = []
uid = 0

for p in parquets:
    df = pd.read_parquet(p)

    text_col = "normalized" if "normalized" in df.columns else ("text" if "text" in df.columns else "verbatim")
    if text_col not in df.columns or "audio" not in df.columns:
        continue

    for _, r in df.iterrows():
        if len(rows) >= TARGET_UTTS:
            break

        txt = str(r[text_col]).strip()
        if not txt:
            continue

        d = to_float_safe(r.get("duration", None), None) if "duration" in df.columns else None
        if d is not None and (d < 1.0 or d > 12.0):
            continue

        snr = to_float_safe(r.get("snr", None), None) if "snr" in df.columns else None
        if snr is not None and snr < 10:
            continue

        cer = to_float_safe(r.get("cer", None), None) if "cer" in df.columns else None
        if cer is not None and cer > 0.12:
            continue

        a = r["audio"]
        if not isinstance(a, dict):
            continue
        b = a.get("bytes", None)
        if not isinstance(b, (bytes, bytearray)):
            continue

        ext = ".wav"
        ap = a.get("path", "")
        if isinstance(ap, str):
            e = os.path.splitext(ap)[1].lower()
            if e in [".wav", ".flac", ".ogg", ".mp3", ".m4a"]:
                ext = e

        out_wav = os.path.join(audio_dir, f"ta_{uid:07d}{ext}")
        uid += 1
        with open(out_wav, "wb") as f:
            f.write(b)

        txt = " ".join(txt.replace("|", " ").split())
        rows.append((out_wav, txt))

    if len(rows) >= TARGET_UTTS:
        break

print("kept utterances:", len(rows))


shards: 7
kept utterances: 1556


In [34]:
# Rebuild clean Tamil audio/filelists from parquet (decode + resample to 22.05k)
import os, glob, random, re
import numpy as np
import soundfile as sf
from datasets import load_dataset, Audio

random.seed(42)

def to_float_safe(x, default=None):
    if x is None:
        return default
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    m = re.search(r"-?\d+(?:\.\d+)?", s)  # handles "tensor(0.)"
    return float(m.group(0)) if m else default

parquets = sorted(glob.glob("/kaggle/working/ivr_tamil/Tamil/train-*.parquet"))
print("parquets:", len(parquets))
assert parquets, "No parquet files found"

# load parquet dataset and let datasets decode audio
ds = load_dataset("parquet", data_files={"train": parquets}, split="train")
print(ds)
print("columns:", ds.column_names)

assert "audio" in ds.column_names, "audio column missing"
ds = ds.cast_column("audio", Audio(sampling_rate=22050))

audio_dir = "/kaggle/working/hi_ta_matcha/audio_ta_22k_clean"
list_dir = "/kaggle/working/Matcha-TTS/data/filelists"
os.makedirs(audio_dir, exist_ok=True)
os.makedirs(list_dir, exist_ok=True)

TARGET_UTTS = 2500   # adjust up/down for Kaggle time+disk
rows = []
uid = 0
bad = 0

text_col = "normalized" if "normalized" in ds.column_names else ("text" if "text" in ds.column_names else "verbatim")

for ex in ds:
    if len(rows) >= TARGET_UTTS:
        break

    txt = str(ex.get(text_col, "")).strip()
    if not txt:
        continue

    d = to_float_safe(ex.get("duration", None), None)
    if d is not None and (d < 1.0 or d > 12.0):
        continue

    snr = to_float_safe(ex.get("snr", None), None)
    if snr is not None and snr < 10:
        continue

    cer = to_float_safe(ex.get("cer", None), None)
    if cer is not None and cer > 0.12:
        continue

    try:
        a = ex["audio"]
        y = np.asarray(a["array"], dtype=np.float32)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if y.size == 0:
            bad += 1
            continue

        out_wav = os.path.join(audio_dir, f"ta22k_{uid:07d}.wav")
        uid += 1
        sf.write(out_wav, y, 22050, subtype="PCM_16")
        txt = " ".join(txt.replace("|", " ").split())
        rows.append((out_wav, txt))
    except Exception:
        bad += 1

print("usable:", len(rows), "bad:", bad)


parquets: 7
Dataset({
    features: ['text', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'audio', 'utterance_pitch_mean', 'utterance_pitch_std', 'snr', 'c50', 'speaking_rate', 'cer', 'duration'],
    num_rows: 2009
})
columns: ['text', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'audio', 'utterance_pitch_mean', 'utterance_pitch_std', 'snr', 'c50', 'speaking_rate', 'cer', 'duration']
usable: 1556 bad: 0


In [40]:
# Split filelists
import random
random.seed(42)
random.shuffle(rows)

val_n = max(300, min(2000, int(0.02 * len(rows))))
val_rows = rows[:val_n]
train_rows = rows[val_n:]

train_out = "/kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt"
val_out = "/kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt"

with open(train_out, "w", encoding="utf-8") as f:
    for ap, tx in train_rows:
        f.write(f"{ap}|{tx}\n")
with open(val_out, "w", encoding="utf-8") as f:
    for ap, tx in val_rows:
        f.write(f"{ap}|{tx}\n")

print("train:", len(train_rows), train_out)
print("val  :", len(val_rows), val_out)


train: 1256 /kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt
val  : 300 /kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt


In [41]:
from pathlib import Path

cfg = """_target_: matcha.data.text_mel_datamodule.TextMelDataModule
name: tamil_ivr
train_filelist_path: /kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt
valid_filelist_path: /kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt
batch_size: 8
num_workers: 2
pin_memory: true
cleaners: [basic_cleaners]
add_blank: true
n_spks: 1
n_fft: 1024
n_feats: 80
sample_rate: 22050
hop_length: 256
win_length: 1024
f_min: 0
f_max: 8000
data_statistics:
  mel_mean: -5.5
  mel_std: 2.0
seed: 42
load_durations: false
"""

p = Path("/kaggle/working/Matcha-TTS/configs/data/tamil.yaml")
p.write_text(cfg, encoding="utf-8")
print("written:", p)


written: /kaggle/working/Matcha-TTS/configs/data/tamil.yaml


In [42]:
from pathlib import Path

symbols_py = Path("/kaggle/working/Matcha-TTS/matcha/text/symbols.py")
train_txt = Path("/kaggle/working/Matcha-TTS/data/filelists/ta_train.txt")
val_txt = Path("/kaggle/working/Matcha-TTS/data/filelists/ta_val.txt")

def collect_chars(paths):
    out = set()
    for p in paths:
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                if "|" not in line:
                    continue
                _, text = line.rstrip("\n").split("|", 1)
                for ch in text:
                    if ch not in "\n\r\t ":
                        out.add(ch)
    return sorted(out)

chars = collect_chars([train_txt, val_txt])

src = symbols_py.read_text(encoding="utf-8")
force_block = (
    "\n\n# --- Tamil force symbols (auto-added) ---\n"
    f"_TAMIL_FORCE = {repr(chars)}\n"
    "for _c in _TAMIL_FORCE:\n"
    "    if _c not in symbols:\n"
    "        symbols.append(_c)\n"
)

if "# --- Tamil force symbols (auto-added) ---" not in src:
    src += force_block
else:
    # replace old force block
    src = src.split("# --- Tamil force symbols (auto-added) ---")[0] + force_block

symbols_py.write_text(src, encoding="utf-8")
print("Force-patched symbols.py with", len(chars), "Tamil chars")


Force-patched symbols.py with 46 Tamil chars


In [43]:
import matcha.text as t
print(type(t.symbols), len(t.symbols), "அ" in t.symbols)


<class 'list'> 224 True


In [44]:
# 2) compute mel stats (important)
%cd /kaggle/working/Matcha-TTS
!matcha-data-stats -i tamil.yaml


/kaggle/working/Matcha-TTS
{'mel_mean': -5.310211658477783, 'mel_std': 2.6833531856536865}                 


In [46]:
# update stats in tamil.yaml
from pathlib import Path
p = Path("/kaggle/working/Matcha-TTS/configs/data/tamil.yaml")
t = p.read_text(encoding="utf-8")
t = t.replace("mel_mean: -5.5", "mel_mean: -5.310211658477783")
t = t.replace("mel_std: 2.0", "mel_std: 2.6833531856536865")
p.write_text(t, encoding="utf-8")
print("updated:", p)


updated: /kaggle/working/Matcha-TTS/configs/data/tamil.yaml


In [49]:
# Verify symbol count after restart
from matcha.text import symbols
N_VOCAB = len(symbols)
print("N_VOCAB =", N_VOCAB)
print("contains அ:", "அ" in symbols)


N_VOCAB = 224
contains அ: True


In [52]:
from pathlib import Path

p = Path("/kaggle/working/Matcha-TTS/matcha/utils/utils.py")
s = p.read_text(encoding="utf-8")

old = """def save_figure_to_numpy(fig):
    data = np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")
    data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    return data
"""

new = """def save_figure_to_numpy(fig):
    fig.canvas.draw()
    if hasattr(fig.canvas, "tostring_rgb"):
        data = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        data = data.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        return data
    rgba = np.asarray(fig.canvas.buffer_rgba())
    return np.ascontiguousarray(rgba[..., :3])
"""

if old in s:
    s = s.replace(old, new)
    p.write_text(s, encoding="utf-8")
    print("Patched:", p)
else:
    print("Exact snippet not found. Show function block and patch manually:")
    print(p)


Patched: /kaggle/working/Matcha-TTS/matcha/utils/utils.py


In [66]:
from pathlib import Path

p = Path("/kaggle/working/Matcha-TTS/matcha/train.py")
s = p.read_text()
old = 'trainer.fit(model=model, datamodule=datamodule, ckpt_path=cfg.get("ckpt_path"))'
new = 'trainer.fit(model=model, datamodule=datamodule, ckpt_path=cfg.get("ckpt_path"), weights_only=False)'

if old in s:
    p.write_text(s.replace(old, new))
    print("patched train.py: weights_only=False")
else:
    print("already patched or pattern changed")


already patched or pattern changed


In [67]:
%cd /kaggle/working/Matcha-TTS
!ls -lh /kaggle/working/ckpts/tamil

!python matcha/train.py \
  experiment=ljspeech \
  data=tamil \
  model.n_vocab=224 \
  trainer.devices=1 \
  trainer.accelerator=gpu \
  trainer.precision=16-mixed \
  trainer.max_epochs=60 \
  trainer.check_val_every_n_epoch=1 \
  extras.print_config=false \
  ~callbacks.model_summary \
  ~callbacks.rich_progress_bar \
  +trainer.log_every_n_steps=100 \
  callbacks.model_checkpoint.dirpath=/kaggle/working/ckpts/tamil \
  "callbacks.model_checkpoint.filename='ta_epoch_{epoch:02d}'" \
  callbacks.model_checkpoint.save_last=true \
  callbacks.model_checkpoint.save_top_k=5 \
  "callbacks.model_checkpoint.monitor='loss/val_epoch'" \
  callbacks.model_checkpoint.mode=min \
  callbacks.model_checkpoint.every_n_epochs=1 \
  callbacks.model_checkpoint.verbose=true \
  ckpt_path=/kaggle/working/ckpts/tamil/last.ckpt \
  data.batch_size=8


/kaggle/working/Matcha-TTS
total 1.7G
-rw-r--r-- 1 root root 209M Mar 23 16:10  last.ckpt
-rw-r--r-- 1 root root 209M Mar 23 15:02  last-v1.ckpt
-rw-r--r-- 1 root root 209M Mar 23 15:02 'ta_epoch_epoch=00.ckpt'
-rw-r--r-- 1 root root 209M Mar 23 16:01 'ta_epoch_epoch=51.ckpt'
-rw-r--r-- 1 root root 209M Mar 23 16:03 'ta_epoch_epoch=53.ckpt'
-rw-r--r-- 1 root root 209M Mar 23 16:04 'ta_epoch_epoch=54.ckpt'
-rw-r--r-- 1 root root 209M Mar 23 16:06 'ta_epoch_epoch=56.ckpt'
-rw-r--r-- 1 root root 209M Mar 23 16:10 'ta_epoch_epoch=59.ckpt'
[2026-03-23 16:52:49,538][matcha.utils.utils][INFO] - Enforcing tags! <cfg.extras.enforce_tags=True>
Seed set to 1234
[2026-03-23 16:52:49,544][__main__][INFO] - Instantiating datamodule <matcha.data.text_mel_datamodule.TextMelDataModule>
[2026-03-23 16:52:51,534][__main__][INFO] - Instantiating model <matcha.models.matcha_tts.MatchaTTS>
/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecat

In [ ]:
!ls -lah /kaggle/working/ckpts/tamil


In [ ]:
%cd /kaggle/working/Matcha-TTS
!LAST_CKPT=$(ls -1t /kaggle/working/ckpts/tamil/*.ckpt | head -n 1); \
echo "Resuming from $LAST_CKPT"; \
python matcha/train.py \
  experiment=ljspeech \
  data=tamil \
  model.n_vocab=224 \
  ckpt_path="$LAST_CKPT" \
  trainer.devices=1 \
  trainer.accelerator=gpu \
  trainer.precision=16-mixed \
  trainer.max_epochs=60 \
  trainer.check_val_every_n_epoch=1 \
  trainer.log_every_n_steps=100 \
  extras.print_config=false \
  ~callbacks.model_summary \
  callbacks.model_checkpoint.dirpath=/kaggle/working/ckpts/tamil \
  callbacks.model_checkpoint.filename=ta_epoch_{epoch:03d} \
  callbacks.model_checkpoint.save_last=true \
  callbacks.model_checkpoint.save_top_k=5 \
  callbacks.model_checkpoint.monitor=loss/val_epoch \
  callbacks.model_checkpoint.mode=min \
  callbacks.model_checkpoint.every_n_epochs=1 \
  data.batch_size=8


In [68]:
import torch

ckpt_path = "/kaggle/working/ckpts/tamil/last.ckpt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

# find ModelCheckpoint callback state
mc = None
for k, v in ckpt.get("callbacks", {}).items():
    if "ModelCheckpoint" in k:
        mc = v
        break

if mc is None:
    print("ModelCheckpoint state not found")
else:
    print("best_model_path :", mc.get("best_model_path"))
    print("best_model_score:", float(mc.get("best_model_score")))
    print("last_model_path :", mc.get("last_model_path"))
    print("\nTop-K (sorted, lower is better):")
    best_k = mc.get("best_k_models", {})
    for p, s in sorted(best_k.items(), key=lambda x: float(x[1])):
        print(f"{float(s):.6f}  {p}")


best_model_path : /kaggle/working/ckpts/tamil/ta_epoch_epoch=53.ckpt
best_model_score: 2.6350014209747314
last_model_path : /kaggle/working/ckpts/tamil/last.ckpt

Top-K (sorted, lower is better):
2.635001  /kaggle/working/ckpts/tamil/ta_epoch_epoch=53.ckpt
2.636746  /kaggle/working/ckpts/tamil/ta_epoch_epoch=56.ckpt
2.648159  /kaggle/working/ckpts/tamil/ta_epoch_epoch=59.ckpt
2.648268  /kaggle/working/ckpts/tamil/ta_epoch_epoch=54.ckpt
2.652328  /kaggle/working/ckpts/tamil/ta_epoch_epoch=51.ckpt


In [ ]:
!cp /kaggle/working/ckpts/tamil/ta_epoch_epoch=53.ckpt /kaggle/working/ckpts/tamil/best.ckpt
!zip -r /kaggle/working/tamil_best_and_last.zip \
  /kaggle/working/ckpts/tamil/best.ckpt \
  /kaggle/working/ckpts/tamil/last.ckpt


In [70]:
from pathlib import Path

p = Path("/kaggle/working/Matcha-TTS/matcha/cli.py")
s = p.read_text()

old = "model = MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device)"
new = "model = MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)"

if old in s:
    p.write_text(s.replace(old, new))
    print("patched cli.py")
else:
    print("line not found; open cli.py and patch manually")


patched cli.py


In [72]:
!pip -q install onnxscript onnx onnxruntime


In [74]:
from pathlib import Path

p = Path("/kaggle/working/Matcha-TTS/matcha/onnx/export.py")
s = p.read_text()

if "dynamo=False" not in s:
    s = s.replace(
        "do_constant_folding=True,\n    )",
        "do_constant_folding=True,\n        dynamo=False,\n    )"
    )
    p.write_text(s)
    print("patched: dynamo=False added")
else:
    print("already patched")


patched: dynamo=False added


In [75]:
!grep -n "dynamo=False\\|to_onnx" /kaggle/working/Matcha-TTS/matcha/onnx/export.py


167:    model.to_onnx(
176:        dynamo=False,


In [76]:
%cd /kaggle/working/Matcha-TTS
!python -m matcha.onnx.export \
  /kaggle/working/ckpts/tamil/ta_epoch_epoch=53.ckpt \
  /kaggle/working/exports/tamil_matcha_e53_t5.onnx \
  --n-timesteps 5


/kaggle/working/Matcha-TTS
[🍵] Loading Matcha checkpoint from /kaggle/working/ckpts/tamil/ta_epoch_epoch=53.ckpt
Setting n_timesteps to 5
[!] Loading ta_epoch_epoch=53!
/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
[+] ta_epoch_epoch=53 loaded!
/kaggle/working/Matcha-TTS/matcha/models/components/flow_matching.py:76: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing a tensor of different shape might lead to errors or silently give incorrect results.
  for step in range(1, len(t_span)):
/usr/local/lib/python3.12/dist-packages/diffusers/models/attention_processor.py:728: TracerWarning: Converting a tensor to a Pyth

In [77]:
# 1) confirm file exists
!ls -lh /kaggle/working/exports/tamil_matcha_e53_t5.onnx


-rw-r--r-- 1 root root 71M Mar 23 17:16 /kaggle/working/exports/tamil_matcha_e53_t5.onnx


In [78]:
# 2) validate ONNX loads
import onnx, onnxruntime as ort
p = "/kaggle/working/exports/tamil_matcha_e53_t5.onnx"
onnx.checker.check_model(onnx.load(p))
sess = ort.InferenceSession(p, providers=["CPUExecutionProvider"])
print("inputs :", [i.name for i in sess.get_inputs()])
print("outputs:", [o.name for o in sess.get_outputs()])


inputs : ['x', 'x_lengths', 'scales']
outputs: ['mel', 'mel_lengths']


In [86]:
# Patch cli.py so inference text prep matches training
from pathlib import Path
import re

p = Path("/kaggle/working/Matcha-TTS/matcha/cli.py")
s = p.read_text(encoding="utf-8")

# Ensure symbols import exists
s = s.replace(
    "from matcha.text import sequence_to_text, text_to_sequence",
    "from matcha.text import sequence_to_text, text_to_sequence, symbols",
)

new_fn = """
def process_text(i: int, text: str, device: torch.device):
    print(f"[{i}] - Input text: {text}")
    ids, cleaned = text_to_sequence(text, ["basic_cleaners"])
    ids = intersperse(ids, len(symbols))  # match training add_blank=True
    x = torch.tensor(ids, dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    print(f"[{i}] - Cleaned text: {cleaned}")
    return {"x_orig": text, "x": x, "x_lengths": x_lengths, "x_phones": cleaned}
""".strip()

s2 = re.sub(
    r"def process_text\(i: int, text: str, device: torch\.device\):.*?return \{\"x_orig\": text, \"x\": x, \"x_lengths\": x_lengths, \"x_phones\": .*?\}",
    new_fn,
    s,
    flags=re.S
)

p.write_text(s2, encoding="utf-8")
print("patched")


patched


In [82]:
%cd /kaggle/working/Matcha-TTS
!mkdir -p /kaggle/working/vocoder /kaggle/working/exports

# download universal HiFi-GAN checkpoint
!wget -O /kaggle/working/vocoder/g_02500000 \
  https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/g_02500000

# export E2E ONNX (Matcha + vocoder)
!python -m matcha.onnx.export \
  /kaggle/working/ckpts/tamil/ta_epoch_epoch=53.ckpt \
  /kaggle/working/exports/tamil_matcha_e53_e2e_t5.onnx \
  --n-timesteps 5 \
  --vocoder-name hifigan_univ_v1 \
  --vocoder-checkpoint-path /kaggle/working/vocoder/g_02500000


/kaggle/working/Matcha-TTS
--2026-03-23 17:26:59--  https://github.com/shivammehta25/Matcha-TTS-checkpoints/releases/download/v1.0/g_02500000
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/742351561/e3ca1338-0d08-472c-9fa9-7f6d5d846e4d?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-23T18%3A01%3A46Z&rscd=attachment%3B+filename%3Dg_02500000&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-23T17%3A01%3A19Z&ske=2026-03-23T18%3A01%3A46Z&sks=b&skv=2018-11-09&sig=ezQW4RRQBfhJ7os%2B72v4QV7aRKD8%2FXEO3PUAGPM1GVo%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NDI4ODYxOSwibmJmIjoxNzc0Mjg2ODE5LCJwYXRoIj

In [90]:
from pathlib import Path

p = Path("/kaggle/working/Matcha-TTS/matcha/cli.py")
s = p.read_text(encoding="utf-8")

# remove blank-token insertion line we added earlier
s = s.replace("    ids = intersperse(ids, len(symbols))  # match training add_blank=True\n", "")
p.write_text(s, encoding="utf-8")
print("patched: no blank intersperse for this checkpoint")


patched: no blank intersperse for this checkpoint


In [91]:
!python -m matcha.onnx.infer \
  /kaggle/working/exports/tamil_matcha_e53_e2e_t5.onnx \
  --text "வணக்கம் இது ஒரு சோதனை. நான் தமிழ் பேசுகிறேன்." \
  --speaking-rate 1.1 \
  --temperature 0.667 \
  --output-dir /kaggle/working/exports/infer_wav


[0] - Input text: வணக்கம் இது ஒரு சோதனை. நான் தமிழ் பேசுகிறேன்.
[0] - Cleaned text: வணக்கம் இது ஒரு சோதனை. நான் தமிழ் பேசுகிறேன்.
The provided model has the vocoder embedded in the graph.
Generating waveform directly
Writing audio to /kaggle/working/exports/infer_wav/output_1.wav
Inference seconds: 1.019516777998433
Generated wav seconds: 1.5441269841269842
Overall RTF: 0.6602544923303997


In [88]:
import soundfile as sf, numpy as np, glob
w = sorted(glob.glob("/kaggle/working/exports/infer_wav/*.wav"))[-1]
x, sr = sf.read(w)
print(w, sr, x.shape, "peak=", float(np.max(np.abs(x))), "mean_abs=", float(np.mean(np.abs(x))))


/kaggle/working/exports/infer_wav/output_1.wav 22050 (7168,) peak= 0.8264826536178589 mean_abs= 0.06409737086921398


In [2]:
!zip -j /kaggle/working/tamil_artifacts_min.zip \
  /kaggle/working/ckpts/tamil/ta_epoch_epoch=53.ckpt \
  /kaggle/working/ckpts/tamil/last.ckpt \
  /kaggle/working/exports/tamil_matcha_e53_e2e_t5.onnx \
  /kaggle/working/Matcha-TTS/configs/data/tamil.yaml \
  /kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt \
  /kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt


	zip warning: name not matched: /kaggle/working/ckpts/tamil/ta_epoch_epoch=53.ckpt
	zip warning: name not matched: /kaggle/working/ckpts/tamil/last.ckpt
	zip warning: name not matched: /kaggle/working/exports/tamil_matcha_e53_e2e_t5.onnx
	zip warning: name not matched: /kaggle/working/Matcha-TTS/configs/data/tamil.yaml
	zip warning: name not matched: /kaggle/working/Matcha-TTS/data/filelists/ta_train_22k.txt
	zip warning: name not matched: /kaggle/working/Matcha-TTS/data/filelists/ta_val_22k.txt

zip error: Nothing to do! (/kaggle/working/tamil_artifacts_min.zip)
